# NBA Player Position Classifier

Can a model figure out what position an NBA player plays just by looking at their stats?

A center who averages 14 rebounds and 2 assists looks very different statistically from a point guard who averages 10 assists and 3 rebounds. The model's job is to learn those patterns.

This is a **multi-class classification** problem — unlike Titanic (survived/died = 2 classes), here we're predicting one of 5 positions: PG, SG, SF, PF, C.

## What's new compared to Titanic
- 5 output classes instead of 2
- Fetching real data from the web using an API
- A bigger confusion matrix (5×5)
- Seeing where the model gets confused (e.g. PF vs C are similar)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('Libraries loaded!')

## Step 1: Load the Data

We'll use the `nba_api` package to pull real NBA player stats. This is different from Titanic where the data was built into seaborn — here we're fetching it live from the NBA's own data service.

If you don't have it installed yet, run this in your terminal:
```bash
conda activate localai
pip install nba_api
```

We'll pull per-game averages for all players from a recent season and use their position as the label.

In [ ]:
from nba_api.stats.endpoints import leaguedashplayerstats
import time

print('Fetching NBA player stats (this takes ~10 seconds)...')

# Pull per-game stats for the 2023-24 season
stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    per_mode_simple='PerGame'
)

df_raw = stats.get_data_frames()[0]
print(f'Fetched {len(df_raw)} players')
print(f'Columns: {list(df_raw.columns[:10])}...')
df_raw.head()

## Step 2: Get Player Positions

The stats endpoint doesn't include position, so we fetch that separately from the player info endpoint and merge the two tables together.

**Merging** joins two tables on a shared column — here `PLAYER_ID`. Think of it like a VLOOKUP in Excel.

In [ ]:
from nba_api.stats.static import players

# Get all player info including position
all_players = players.get_players()
players_df = pd.DataFrame(all_players)

# The static endpoint doesn't have position — use CommonPlayerInfo for active players
# Instead, we'll use a built-in position mapping from the draft combine or team rosters
# Simplest approach: use the position from leaguedashplayerstats with position filter

positions = ['PG', 'SG', 'SF', 'PF', 'C']
dfs = []

for pos in positions:
    print(f'  Fetching {pos}...')
    result = leaguedashplayerstats.LeagueDashPlayerStats(
        season='2023-24',
        per_mode_simple='PerGame',
        player_position_abbreviation_nullable=pos
    )
    temp = result.get_data_frames()[0]
    temp['POSITION'] = pos
    dfs.append(temp)
    time.sleep(0.6)  # be polite to the API

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(df)}')
print(df['POSITION'].value_counts())

## Step 3: Explore the Data

Before training, look at whether the stats actually differ by position. If they don't, no model will be able to predict position accurately.

You'd expect:
- **PG** → high assists, low rebounds
- **C** → high rebounds, low assists, high blocks
- **SG/SF** → high points, moderate everything else

In [ ]:
# Filter to players with meaningful minutes (avoid noise from rarely-used players)
df = df[df['GP'] >= 20].copy()
print(f'Players with 20+ games: {len(df)}')
print()

# Average stats by position
key_stats = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'TOV']
df.groupby('POSITION')[key_stats].mean().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, stat in zip(axes, ['AST', 'REB', 'BLK']):
    df.groupby('POSITION')[stat].mean().sort_values().plot(
        kind='bar', ax=ax, color='steelblue'
    )
    ax.set_title(f'Avg {stat} by Position')
    ax.set_ylabel(stat)
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Step 4: Prepare Features

We pick the stats most likely to distinguish positions. We don't need all columns — many are redundant (e.g. FGM and FGA both measure shooting volume).

We also need to convert position labels ("PG", "C", etc.) to numbers — sklearn models only work with numbers. `LabelEncoder` does this automatically.

In [ ]:
features = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'TOV', 
            'FG_PCT', 'FG3_PCT', 'FT_PCT', 'MIN']

X = df[features].fillna(0)
y_raw = df['POSITION']

# Convert position strings to numbers: C=0, PF=1, PG=2, SF=3, SG=4
le = LabelEncoder()
y = le.fit_transform(y_raw)

print('Position encoding:')
for i, pos in enumerate(le.classes_):
    print(f'  {i} = {pos}')

print(f'\nFeatures shape: {X.shape}')

## Step 5: Train and Evaluate

Same Random Forest approach as Titanic — it works well for tabular data with multiple classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y ensures each position is proportionally represented in both splits
)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, preds):.1%}')
print()
print(classification_report(y_test, preds, target_names=le.classes_))

The **classification report** breaks accuracy down per position:
- **Precision** — of all players it predicted as PG, what % actually were PG?
- **Recall** — of all actual PGs, what % did it correctly identify?
- **F1** — the average of precision and recall, one combined score

You'll likely see C and PG score highest — they have the most distinctive stats. SG and SF will probably score lowest — they're the most statistically similar positions.

## Step 6: Confusion Matrix

Now it's 5×5 instead of 2×2. Each row is the actual position, each column is what the model predicted. Numbers on the diagonal = correct. Off-diagonal = mistakes.

Look for which positions get confused with each other — that's the interesting part.

In [ ]:
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — NBA Position Prediction')
plt.ylabel('Actual Position')
plt.xlabel('Predicted Position')
plt.tight_layout()
plt.show()

## Step 7: Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — What Stats Predict Position Best?')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Step 8: Predict a Specific Player's Position

Enter any stat line and see what position the model thinks that player is.

In [ ]:
# Try a classic center stat line
# PTS  REB  AST  BLK  STL  TOV  FG%   3P%   FT%   MIN
player = pd.DataFrame([{
    'PTS': 18.0, 'REB': 12.0, 'AST': 2.0, 'BLK': 2.5, 'STL': 0.8,
    'TOV': 2.5, 'FG_PCT': 0.56, 'FG3_PCT': 0.0, 'FT_PCT': 0.65, 'MIN': 32.0
}])

pred = rf.predict(player)[0]
probs = rf.predict_proba(player)[0]

print(f'Predicted position: {le.classes_[pred]}')
print()
print('Probability for each position:')
for pos, prob in zip(le.classes_, probs):
    bar = '█' * int(prob * 30)
    print(f'  {pos}: {bar} {prob:.1%}')
print()
print('Try changing AST to 10.0 and REB to 4.0 — does it predict PG?')

## What You Built

A multi-class classifier that predicts NBA position from stats — and along the way you learned:
- How to fetch real data from a live API
- How to merge two datasets on a shared column
- How `LabelEncoder` converts text labels to numbers
- How to read a 5×5 confusion matrix
- Precision vs recall vs F1

## Things to Try

1. **Change the season** — try `'2022-23'` or `'2015-16'` and see if positions were more distinct in earlier eras
2. **Add more features** — usage rate, plus/minus, pace
3. **Collapse positions** — combine PG+SG into "Guard" and PF+C into "Big" for a simpler 3-class problem
4. **Find a specific real player** — look up LeBron's 2023-24 stats and see what position the model assigns him